

# Selective memory: Creating smart agents with Elasticsearch managed memory

This Notebook is the supporting content for the article: [Selective memory: creating smart agents with Elasticsearch managed memory](https://www.elastic.co/search-labs/blog/smarter-agents-with-memory)

We are going to use OpenAI Response API and tools to create an agent that can remember things across conversations selectively. This means based on who is talking with it will use Elasticsearch Document Level Security to retrieve the right memories.

In [ ]:
%pip install openai elasticsearch==9.1.0 --quiet

## Initializing OpenAI and Elasticsearch

In [ ]:
from openai import OpenAI
from elasticsearch import Elasticsearch
import getpass

# Initialize OpenAI
client = OpenAI(api_key=getpass.getpass("OpenAI API KEY"))

# Elasticsearch configuration
ELASTICSEARCH_URL = api_key = getpass.getpass("ELASTICSEARCH_URL")
ELASTICSEARCH_API_KEY = api_key = getpass.getpass("ELASTICSEARCH_API_KEY")
ELASTICSEARCH_INDEX = "memories"

# Initialize Elasticsearch client
es_client = Elasticsearch(hosts=[ELASTICSEARCH_URL], api_key=ELASTICSEARCH_API_KEY)

In [ ]:
print(es_client.info())

## Creating memories index

In [ ]:
mappings = {
    "properties": {
        "user_id": {"type": "keyword"},
        "memory_type": {"type": "keyword"},
        "created_at": {"type": "date"},
        "memory_text": {
            "type": "text",
            "fields": {"semantic": {"type": "semantic_text"}},
        },
    }
}

# Create the index
try:
    es_client.indices.create(index=ELASTICSEARCH_INDEX, mappings=mappings, ignore=400)
    print(f"Index '{ELASTICSEARCH_INDEX}' created successfully")
except Exception as e:
    print(f"Error creating index '{ELASTICSEARCH_INDEX}': {e}")

## Sample memory

In [ ]:
from datetime import datetime

# Sample document based on the mappings
memories = [
    {
        "user_id": "peter125",
        "memory_type": "outie",
        "created_at": datetime.now(),
        "memory_text": "User name is Peter Johnson. His favorite family destination is Disneyland.",
    },
    {
        "user_id": "innie-petter",
        "memory_type": "innie",
        "created_at": datetime.now(),
        "memory_text": "Peter needs to review the Q4 budget spreadsheet before Friday.",
    },
]

operations = []
for mem in memories:
    operations.append({"index": {"_index": "memories"}})
    operations.append(mem)

# Index the sample document
try:
    response = es_client.bulk(operations=operations)
    print(f"Bulk indexed {len(memories)} memories.")
except Exception as e:
    print("Error indexing sample document:", e)

_You may see a timeout the first time because the ML nodes are warming. Wait some minutes and try again_

## Creating roles

In [ ]:
innie_role_descriptor = {
    "indices": [
        {
            "names": ["memories"],
            "privileges": ["read", "write"],
            "query": {"bool": {"filter": [{"term": {"memory_type": "innie"}}]}},
        }
    ]
}

# Create the "innie" role
try:
    response = es_client.security.put_role(name="innie", body=innie_role_descriptor)
    print("Role 'innie' created successfully:", response)
except Exception as e:
    print(f"Error creating role 'innie': {e}")

In [ ]:
outie_role_descriptor = {
    "indices": [
        {
            "names": ["memories"],
            "privileges": ["read", "write"],
            "query": {"bool": {"filter": [{"term": {"memory_type": "outie"}}]}},
        }
    ]
}

# Create the "outie" role
try:
    response = es_client.security.put_role(name="outie", body=outie_role_descriptor)
    print("Role 'outie' created successfully:", response)
except Exception as e:
    print(f"Error creating role 'outie': {e}")

## Creating users with roles

In [ ]:
# Define the inne user
user_data = {
    "password": "YourSecurePassword123",  # Choose a secure password
    "roles": ["innie"],  # Assign the 'innie user' role
    "full_name": "Peter Peter Johnson",  # User's full name
    "email": "peter125@example.com",
}

# Create the user
response = es_client.security.put_user(username="peter125", body=user_data)

print("User creation response:", response)

In [ ]:
# Define the outie user
user_data = {
    "password": "YourSecurePassword123",  # Choose a secure password
    "roles": ["outie"],  # Assign the 'outie user' role
    "full_name": "Jennifer Connor",  # User's full name
    "email": "jennifer321@example.com",
}

# Create the user
response = es_client.security.put_user(username="jennifer321", body=user_data)

print("User creation response:", response)

## Testing roles

In [ ]:
user_es_client = Elasticsearch(
    hosts=[ELASTICSEARCH_URL], basic_auth=("peter125", "YourSecurePassword123")
)

response = user_es_client.search(index="memories", query={"match_all": {}})
# Memory show up because peter125 is an innie
print(response["hits"]["hits"])

In [ ]:
user_es_client = Elasticsearch(
    hosts=[ELASTICSEARCH_URL], basic_auth=("jennifer321", "YourSecurePassword123")
)

response = user_es_client.search(index="memories", query={"match_all": {}})
# Jennifer is an outie so Mark wont have innie memories when talking to her
print(response["hits"]["hits"])

## Defining tools

In [ ]:
tools = [
    {
        "type": "function",
        "name": "GetKnowledge",
        "description": "Search for relevant documents based on the internal knowledge of the agent.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The natural language query to search the vector database.",
                }
            },
            "required": ["query"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "GetMemories",
        "description": "Get memories from previous conversations to answer the question.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The natural language query to search the vector database.",
                }
            },
            "required": ["query"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "SetMemory",
        "description": "Save a memory if there is something relevant to remember in the query",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The natural language query to search the vector database.",
                }
            },
            "required": ["query"],
            "additionalProperties": False,
        },
    },
]

## Creating tool functions

In [ ]:
# helper to format the output
def build_tool_response(call_id: str, result: str):
    return {"type": "function_call_output", "call_id": call_id, "output": str(result)}

In [ ]:
def get_knowledge(query: str):
    return "Empty knowledgebase"

In [ ]:
def get_memory(query: str):
    user_es_client = Elasticsearch(
        hosts=[ELASTICSEARCH_URL], basic_auth=("peter125", "YourSecurePassword123")
    )

    # Note you don't have to worry about applying security filters to the query because they are handled by Elasticsearch document-level security

    es_query = {
        "retriever": {
            "rrf": {
                "retrievers": [
                    {
                        "standard": {
                            "query": {
                                "semantic": {"field": "semantic_field", "query": query}
                            }
                        }
                    },
                    {
                        "standard": {
                            "query": {
                                "multi_match": {
                                    "query": query,
                                    "fields": ["memory_text"],
                                }
                            }
                        }
                    },
                ],
                "rank_window_size": 50,
                "rank_constant": 20,
            }
        }
    }

    response = user_es_client.search(index=ELASTICSEARCH_INDEX, body=es_query)

    response_str = "Memories\n"
    for hit in response["hits"]["hits"]:
        response_str += (
            f"{hit['_source']['user_id']}: (${hit['_source']['memory_text']})\n"
        )

    print(response_str)

    return response_str

In [ ]:
print(get_memory("What's my favorite family destination?"))

In [ ]:
def set_memory(query: str, call_id):
    return f"Memory for {query} was saved."

# Handling tools response

The response will be a list of tool calls, we must call each of the tools, and call the LLM back with the rool responses to generate the answer.

In [ ]:
def process_tool_calls(response, messages):
    """
    Processes tool calls from an OpenAI response and appends results to the message list.
    """
    for tool_call in response.output:
        if getattr(tool_call, "type", None) != "function_call":
            continue  # Skip anything that's not a function call

        tool_name = getattr(tool_call, "name", None)
        call_id = getattr(tool_call, "call_id", None)
        args = getattr(tool_call, "arguments", "{}")

        # Parse JSON safely
        try:
            import json

            args_dict = json.loads(args)
        except Exception as e:
            print(f"Error parsing arguments: {e}")
            args_dict = {}

        # Route to correct function
        if tool_name == "GetMemories":
            result = get_memory(args_dict.get("query", ""))
        elif tool_name == "SetMemory":
            result = set_memory(args_dict.get("query", ""))
        elif tool_name == "GetKnowledge":
            result = get_knowledge(args_dict.get("query", ""))
        else:
            result = f"Unknown tool: {tool_name}"

        # Build the formatted tool response
        tool_response = {
            "type": "function_call_output",
            "call_id": call_id,
            "output": str(result),
        }

        # Append to the conversation history
        messages.append(
            {
                "role": "assistant",
                "content": [{"type": "output_text", "text": json.dumps(tool_response)}],
            }
        )

        print(f"Executed {tool_name} → {result}")

    return messages

## Creating the agent

In [ ]:
question = "Whats my favorite family destination?"

messages = [
    {
        "role": "system",
        "content": "Use GetKnowledge and GetMemories ONCE to grab context first.",
    },
    {"role": "user", "content": question},
]

response = client.responses.create(
    model="gpt-4.1-mini", input=messages, tools=tools, parallel_tool_calls=True
)

print(response.output)

In [ ]:
import pandas as pd

# Create a list to store the tool call and function call details
tool_calls = []

# Iterate through the response output and collect the details
for i in response.output:
    tool_calls.append(
        {
            "Type": i.type,
            "Arguments": i.arguments if hasattr(i, "arguments") else "N/A",
            "Output": str(i.output) if hasattr(i, "output") else "N/A",
            "Name": i.name if hasattr(i, "name") else "N/A",
        }
    )

# Convert the list to a DataFrame for tabular display
df_tool_calls = pd.DataFrame(tool_calls)

# Display the DataFrame
df_tool_calls

In [ ]:
final_messages = process_tool_calls(response, messages)

print("Messages")
print(messages)

In [ ]:
final_response = client.responses.create(model="gpt-4.1-mini", input=final_messages)

print(final_response.output[0].content[0].text)